# Lab 06: How to build an evaluation set for RAG

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | In class**

## Goal

Build a fixed set of questions and use it to compare retrieval, answers and refusals one question at a time. You will
see a wrong duration score higher on word overlap than a correct sentence, then inspect the answers behind a pipeline's
aggregate scores.

Lab 04 helped us diagnose individual failures. Lab 05 rebuilt the pipeline with LangChain. Before changing that
pipeline, we need a repeatable way to see what improves and what breaks. The test questions, expected answers and
scoring rules give us that comparison. They also need inspection: a mistaken answer key can reward the wrong behaviour.

Parts A–D build the measures and ten evaluation records, then keep a row for every reply. The self-study sections
version the records, report the scores with their denominators and examine policy failures in a larger set.

## What you will learn

- Compute hit@3 and mean reciprocal rank (MRR@3), including misses in the denominator.
- Compute exact match (EM) and token F1 against accepted answers, and explain their limits on generated sentences.
- Write golden records with known support and accepted answers or an expected refusal, then version the set.
- Score a pipeline one question at a time and preserve the relationship between each question and its result.
- Report answer checks and refusals separately, and inspect each kind of question as its own slice.

**In class:** Parts A to D, except the versioning subsection at the end of Part C. That subsection and Parts E and F
are self-study.


## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-28
OPENAI_API_KEY    found in environment


The settings above identify the chat and embedding models and their endpoints. `JSON_MODE` in `models.env` chooses
how the pipeline requests structured answers. Running through Part F embeds the corpus once (about 21,000 tokens),
embeds the search questions and makes 65 chat calls: one demonstration, ten initial records, 38 shared records and
16 policy repeats. Completing Challenge Task 1 adds six chat calls and six searches. Re-running cells makes new calls;
charges depend on your provider, model and token usage.

The saved outputs use `gpt-4o-mini` and `text-embedding-3-small`. Treat the checkpoints as a reading of those outputs.
Your replies and similarity scores can vary; inspect the rankings and facts before comparing totals.

An **evaluation set**, or **golden set**, is a fixed list of questions with an answer key: accepted answers, known
sources or a required refusal. The key stays outside the searchable corpus. Only the question and retrieved evidence
reach the answering model; the evaluator reads the key afterwards. These records are tests, rather than examples added
to the prompt.

Running the same tests after a pipeline change is **regression testing**. It lets us notice a previously working
case that the change breaks. This lab uses development records that we may inspect and revise; the protected
questions are not loaded or scored.


---

## Part A: Evaluate Retrieval Rankings and Answer Similarity by Hand

Part A works each measure on numbers small enough to check in your head: first the retrieval measures, which read the
ranked sources, then the answer measures, which read the answer text.

A question's **known support** is the source its reference answer comes from: for a development question, the
paragraph it was written from (`S03-P00`); for a policy question, the policy (`H02`). A source label identifies a
paragraph or policy. It does not certify that every chunk from that source contains the complete answer, and it may
omit another source that could answer the question.

**hit@3** asks, for each question, whether a known source is among the top three results, and counts the questions
where it is:

$$
\text{hit@3} = \frac{\text{number of questions with a known source in the top 3}}{\text{number of questions}}
$$

- hit: a known source was retrieved.
- @3: only the top 3 results are read.
- the denominator: every question asked, the misses included.

hit@3 says whether the support came back, not how high. The **reciprocal rank** (RR) of one question says how high:

$$
\text{RR} = \begin{cases} \dfrac{1}{\text{rank of the first known source}} & \text{if a known source is in the top 3} \\[1ex] 0 & \text{otherwise} \end{cases}
$$

- rank 1 gives 1, rank 2 gives 0.5, rank 3 gives about 0.333: support found later is worth less.
- a miss gives 0, not an empty value, so it still pulls the average down.

The **mean reciprocal rank** (MRR; MRR@3 here, because only three ranks are read) is the average over the questions:

$$
\text{MRR@3} = \frac{1}{|Q|} \sum_{q \in Q} \text{RR}_q
$$

- $Q$: the questions asked; $|Q|$: how many there are, the misses included.
- $\text{RR}_q$: the reciprocal rank of question $q$.

Take three questions whose first known source comes at rank 1, at rank 3 and not at all. We already know their ranks,
so this example makes no API call. Each position in `ranks` represents one question. `None` means its known support
was absent from the three inspected results; it does not mean we forgot to score that question:



In [3]:
# ranks: the rank of the first known source for three questions; None means no known source in the top 3
# reciprocal: each question's RR, 1 / rank, or 0 for the miss
ranks = [1, 3, None]
reciprocal = [1 / rank if rank else 0 for rank in ranks]
found_any = sum(rank is not None for rank in ranks)
print("reciprocal ranks:", [round(r, 3) for r in reciprocal])
print(f"hit@3: {found_any} of {len(ranks)}")

reciprocal ranks: [1.0, 0.333, 0]
hit@3: 2 of 3


- `1 / rank if rank else 0`: Python reads `None` as false, so the miss gets 0.
- `sum(rank is not None for rank in ranks)`: each `True` counts as 1, so this counts the questions that found a known
  source.

Now MRR@3, twice: once divided by all three questions, as the formula says, and once divided by only the two that
found their support:

The resulting list still has three positions, including a zero for the miss. Keeping that position is what lets the
next calculation average over the questions we actually asked.


In [4]:
# MRR@3 with two denominators: every question asked, and only the questions that found their support
print(f"MRR@3: {sum(reciprocal) / len(ranks):.3f}  (divided by all {len(ranks)} questions)")
print(f"MRR@3: {sum(reciprocal) / found_any:.3f}  (divided by the {found_any} that found something)")

MRR@3: 0.444  (divided by all 3 questions)
MRR@3: 0.667  (divided by the 2 that found something)


Both lines run, and only the first is MRR: (1 + 0.333 + 0) / 3 = 0.444. Dividing by the two questions that found
their support gives 0.667, and the miss disappears. **Keep every question you asked in the denominator, and print the
denominator beside the result.**

Now the answers. Both answer measures come from **SQuAD** (2016), a reading benchmark whose answers are short spans
copied from a paragraph; our 30 development questions are a sample of it. Before two texts are compared, both are
**normalised**, as SQuAD's scoring script does:

1. convert to lower case;
2. remove ASCII punctuation;
3. remove the articles *a*, *an* and *the*;
4. collapse whitespace to single spaces.

`string.punctuation` is Python's list of ASCII punctuation marks. `re.sub(r"\b(a|an|the)\b", " ", text)` replaces each
article that stands as a whole word (`\b` marks the edge of a word) with a space, and `" ".join(text.split())` turns
any run of whitespace into one space. Punctuation is deleted, rather than replaced: a hyphenated word can be joined
into one word. Curly quotation marks are outside `string.punctuation` and survive. We keep this benchmark convention
so we know exactly which text comparison we are teaching:


In [5]:
# normalize(): SQuAD's normalisation, tried on two texts
import re
import string

def normalize(text):
    text = "".join(c for c in text.lower() if c not in string.punctuation)
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", text).split())

print(normalize("About 30 minutes."), "|", normalize("The Denver Broncos"))

about 30 minutes | denver broncos


The capital letters, the full stop and *The* are gone.

**Exact match** (EM) compares the two normalised texts as a whole:

$$
\text{EM}(a, r) = \begin{cases} 1 & \text{if } \text{normalize}(a) = \text{normalize}(r) \\ 0 & \text{otherwise} \end{cases}
$$

- $a$: the answer; $r$: the reference answer.
- an extra word that survives normalisation makes EM 0.

**Token F1** gives partial credit for the words the two texts share. Here a token is a word produced by `.split()`
after normalisation. These are not the subword tokens the model API bills for. Take the answer *about 30 minutes* and the
reference *30 minutes*, and count the shared words first. `Counter`, from Python's `collections`, counts how often each
word appears in a list. `&` between two counters keeps the words they share, each with the smaller of its two counts:
a word twice in the answer and once in the reference is shared once. The total of those counts is the **overlap**:


In [6]:
# answer_words, reference_words: the two texts after normalising, as lists of words
# common: the words both lists hold, each as many times as it appears in both; overlap: how many words that is
from collections import Counter

answer_words = normalize("about 30 minutes").split()
reference_words = normalize("30 minutes").split()
common = Counter(answer_words) & Counter(reference_words)
print(Counter(answer_words), "&", Counter(reference_words))
print("=", common)
overlap = sum(common.values())
print("overlap:", overlap)

Counter({'about': 1, '30': 1, 'minutes': 1}) & Counter({'30': 1, 'minutes': 1})
= Counter({'30': 1, 'minutes': 1})
overlap: 2


*30* and *minutes* are shared; *about* is not. From the overlap come three numbers:

$$
P = \frac{\text{overlap}}{\text{words in the answer}} \qquad R = \frac{\text{overlap}}{\text{words in the reference}} \qquad F_1 = \frac{2 \cdot P \cdot R}{P + R}
$$

- $P$, **precision**: the share of the answer's words that are in the reference. Extra words in the answer lower it. 
- $R$, **recall**: the share of the reference's words that are in the answer. Words missing from the answer lower it.
- $F_1$: the **harmonic mean** of the two, which stays low when either one is low.


*Predict first:* with two shared words, three answer words and two reference words, what are $P$, $R$ and F1?

__Precision__: Of the words the system generated, how many match the reference?
$$
P = \frac{2}{3}
$$
__Recall__: Of the words in the reference, how many did the system include?
$$
R = \frac{2}{2}
$$
__F1__: Combines precision and recall into one score using their harmonic mean.
\[
\text{F1}=\frac{2PR}{P+R}=0.800
\]
$$
F_1 = \frac{2PR}{P+R} = 0.80
$$

In [7]:
# precision, recall and F1 from the overlap; EM compares the two word lists as a whole
precision = overlap / len(answer_words)
recall = overlap / len(reference_words)
f1 = 2 * precision * recall / (precision + recall)
print(f"precision {precision:.3f} | recall {recall:.3f} | F1 {f1:.3f} | EM {int(answer_words == reference_words)}")

precision 0.667 | recall 1.000 | F1 0.800 | EM 0


Two of the three answer words are in the reference (precision 0.667), and both reference words are in the answer
(recall 1.000): F1 is 2 × 0.667 × 1 / 1.667 = 0.800. EM is 0, because of the one extra word.

### Lesson to Remember

RAG evaluation measures two different stages of the pipeline:

- **Retrieval quality:** Hit@3 measures whether the correct evidence was retrieved, while MRR measures how highly it ranked.
- **Answer quality:** Exact Match (EM) checks whether the generated answer matches the reference exactly (after normalization), while token F1 measures how many words they share.

**But a high evaluation score does not always mean the answer is factually correct.**

Consider the question: *How long does the reset link remain valid?*

Assume the correct answer is **30 minutes**.

| Generated answer | Reference answer | F1 |
|---|---|---|
| About 30 minutes | 30 minutes | 0.800 |
| 60 minutes | 30 minutes | 0.500 |
| 30 minutes | 30 minutes | 1.000 |

Notice that **"60 minutes" is factually wrong**, yet it receives an F1 score of 0.500 simply because it shares the word *minutes* with the reference.

**The key takeaway:** Hit@3 and MRR evaluate retrieval rankings, while EM and F1 evaluate text agreement. None of these metrics alone guarantees that the generated answer is factually correct.

In Part B, we'll implement these metrics as reusable Python functions so we can evaluate our RAG pipeline consistently.

### Checkpoint A

| You should see | Our run |
|---|---|
| Ranks 1, 3 and none | reciprocal ranks `[1.0, 0.333, 0]`; hit@3 2 of 3; MRR@3 0.444, or 0.667 when the miss leaves the denominator |
| Normalised | `about 30 minutes \| denver broncos` |
| *about 30 minutes* against *30 minutes* | overlap 2; precision 0.667, recall 1.000, F1 0.800, EM 0 |

Both retrieval measures depend on the source labels; both answer measures depend on the reference text. We have
checked the arithmetic on visible inputs. Part B reuses it on examples where the arithmetic is correct but the score
can mislead us about the answer.


---

## Part B: Reuse the measures and test what they can miss

Part B makes the measures reusable and checks them against Part A's numbers before any model answer is scored.

A question can have more than one accepted answer: SQuAD keeps the strings several annotators chose, such as *Miller*
and *Von Miller*. With several references, each measure keeps its best score:

$$
\text{EM}(a, R) = \max_{r \in R} \text{EM}(a, r) \qquad F_1(a, R) = \max_{r \in R} F_1(a, r)
$$

- $R$: the list of accepted answers.
- the maximum, not the average: a right answer should not lose credit because one annotator wrote it differently.

`em_f1()` puts Part A's calculations into a loop over a list of reference strings. It returns a two-item tuple:
`(best_em, best_f1)`, an integer 0 or 1 followed by a float between 0 and 1. The two maxima are kept independently;
we do not need to choose a single winning reference first. Try it on the same input we just calculated:


In [8]:
# em_f1(): the best EM and the best F1 of one answer over all its references
def em_f1(prediction, references):
    best_em, best_f1 = 0, 0.0
    for reference in references:
        a, b = normalize(prediction).split(), normalize(reference).split()
        overlap = sum((Counter(a) & Counter(b)).values())
        f1 = 0.0
        if overlap:
            precision, recall = overlap / len(a), overlap / len(b)
            f1 = 2 * precision * recall / (precision + recall)
        best_em, best_f1 = max(best_em, int(a == b)), max(best_f1, f1)
    return best_em, best_f1

em_f1("about 30 minutes", ["30 minutes"])

(0, 0.8)

### Keep the best score across accepted answers

1. `best_em` and `best_f1` start at 0.
2. For each reference, both texts are normalised and split into words: `a` for the answer, `b` for the reference.
3. `overlap` counts the shared words, as in Part A.
4. `if overlap` comes before any division: with no shared word, or an empty answer, F1 stays 0 instead of dividing by
   zero.
5. With overlap, compute precision, recall and their harmonic mean, as in Part A.
6. `max()` keeps the better EM and the better F1 seen so far, and the pair is returned.

The same 0.800 as by hand. `first_rank()` does the retrieval side: given the retrieved sources in ranked order and a
record's known support, it returns the rank of the first source that is support, or `None`:


In [9]:
# first_rank(): the rank (1 = first) of the first retrieved source that is in the support, or None
def first_rank(sources, support):
    return next((rank for rank, source in enumerate(sources, 1) if source in support), None)

first_rank(["H04", "H01", "H02"], ["H02"])

3

`enumerate(sources, 1)` numbers the sources from 1, and `next(..., None)` takes the first rank whose source is
support, or `None` when there is none. H02 is third: rank 3. The parentheses inside `next()` make a generator: it offers
matching ranks one at a time and stops at the first. The source list must already be in retrieval order.

`first_rank()` reads the list you give it; it does not cut it to three. We call the result a rank at 3 only when the
list contains the top three results. A longer list would measure a different retrieval depth.

Now four constructed answers to score, each against its references. These isolate the metric's behaviour; they are
not model replies. In the loop below, `for prediction, references in cases` unpacks one pair at a time, and
`em, f1 = ...` unpacks the returned tuple. `!r` prints quotes around the answer so we can see its exact text:

> 1. *30 minutes*, reference *30 minutes*
> 2. *30 minutes*, reference *10 minutes*
> 3. *Nikola Tesla was born on July 10, 1856.*, reference *1856*
> 4. *about 30 minutes*, references *30 minutes* or *half an hour*

*Predict first:* which of the four answers is wrong, and which one gets the lowest F1?


In [10]:
# cases: (answer, references) pairs; each line prints EM and F1 for one pair
cases = [
    ("30 minutes", ["30 minutes"]),
    ("30 minutes", ["10 minutes"]),
    ("Nikola Tesla was born on July 10, 1856.", ["1856"]),
    ("about 30 minutes", ["30 minutes", "half an hour"]),
]
for prediction, references in cases:
    em, f1 = em_f1(prediction, references)
    print(f"EM {em} | F1 {f1:.3f} | {prediction!r} against {references}")

EM 1 | F1 1.000 | '30 minutes' against ['30 minutes']
EM 0 | F1 0.500 | '30 minutes' against ['10 minutes']
EM 0 | F1 0.222 | 'Nikola Tesla was born on July 10, 1856.' against ['1856']
EM 0 | F1 0.800 | 'about 30 minutes' against ['30 minutes', 'half an hour']


Read the second and third lines together:

- *30 minutes* against *10 minutes* is wrong, and F1 gives it 0.500: one of its two words, *minutes*, matches.
- The Tesla sentence is right, and F1 gives it 0.222: the reference's one word is in it (recall 1), but so are seven
  words the reference lacks (precision 1 of 8).
- The last line keeps the better of the two references, 0.800, not their average.

**F1 measures shared words, not whether the fact is right: here a wrong short answer scores more than twice a right
sentence.**

**Lesson to Remember**
> On a generated sentence, extra wording can lower EM and F1 even when the fact is right. Shared wording can raise
> F1 when the fact is wrong. Print the answer beside its score and read what the check counted.

### Checkpoint B

| You should see | Our run |
|---|---|
| The functions | `(0, 0.8)` and `3` |
| Four answers | EM 1, F1 1.000; EM 0, F1 0.500 (wrong); EM 0, F1 0.222 (right); EM 0, F1 0.800 |


---

## Part C: Define questions, accepted answers and required refusals

Part C writes the questions the pipeline will be scored on, with what counts as right for each one.

A **golden record** holds one question and the labels the evaluator will use afterwards. A label makes a check
repeatable; it still needs a person to decide whether it expresses the behaviour we want. For Tesla's birth year,
we already know the question, the reference *1856* and the paragraph it comes from. Keep those responsibilities clear:

| Field | Value type and purpose | Does the answering model see it? |
|---|---|---|
| `question_id` | String that keeps the case identifiable when results are reordered | No |
| `split` | String, `development` here: we may inspect these cases while making changes | No |
| `question` | String containing the search and answer request | Yes |
| `type` | String describing the behaviour to group into a slice, such as `exception` | No |
| `known_support` | List of source IDs used to check retrieval, such as `['S03-P00']` | No |
| `references` | List of accepted answer strings for the text checks | No |
| `refuse` | Boolean: whether the expected reply is *Insufficient evidence.* | No |

The retrieved documents are also sent to the answering model. The answer key is not. A refusal case normally has
empty references because an answer comparison is not its test. Later, Part F will also construct a case with support
in the corpus but deliberately absent from the evidence.

A development question already holds most of this. Load the 30 and look at the Tesla one:


In [11]:
# dev: the 30 development questions as the data file holds them; tesla: the one about Tesla's birth year
import json

DATA = ROOT / "data"
dev = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
tesla = next(q for q in dev if q["question"].startswith("In what year was Nikola Tesla born"))
print(len(dev), "questions")
print(json.dumps(tesla, ensure_ascii=False, indent=1))

30 questions
{
 "question_id": "56df9e2838dc4217001520f6",
 "question": "In what year was Nikola Tesla born? ",
 "answers": [
  {
   "answer_start": 54,
   "text": "1856"
  },
  {
   "answer_start": 54,
   "text": "1856"
  },
  {
   "answer_start": 54,
   "text": "1856"
  }
 ],
 "known_support": [
  "S03-P00"
 ],
 "article_index": 3
}


The file gives us a Python list of 30 dictionaries. `next(...)` selects the first whose question starts with the
Tesla wording, and `json.dumps(..., indent=1)` displays that dictionary as readable JSON. Before transforming it,
read the fields we will use:

- `answers`: one entry per annotator, the `text` they marked and where it starts in the paragraph; all three marked
  *1856*. `answer_start: 54` is a character offset into the source paragraph, not into a chunk.
- `known_support`: `S03-P00`, paragraph 0 of article 3, the paragraph the question was written from.
- `article_index`: which of the five Wikipedia articles it comes from.

Build its record by hand: the ID, the text and the support, and the answer strings without repeats. `{a["text"] for a
in ...}` is a set, which keeps each string once; `sorted()` turns it into a list in a fixed order. The three annotator
entries therefore become `['1856']`. We leave out `answer_start` and `article_index`: these measures compare answer
text and source IDs and do not read offsets or article numbers:


In [12]:
# record: one golden record, built by hand from the Tesla question
record = {"question_id": tesla["question_id"], "split": "development", "question": tesla["question"],
          "type": "answerable", "known_support": tesla["known_support"],
          "references": sorted({a["text"] for a in tesla["answers"]}), "refuse": False}
print(record)

{'question_id': '56df9e2838dc4217001520f6', 'split': 'development', 'question': 'In what year was Nikola Tesla born? ', 'type': 'answerable', 'known_support': ['S03-P00'], 'references': ['1856'], 'refuse': False}


`record_from_dev()` is the same dictionary as a function, for any development question. It must give the record
you just built. The equality below compares the dictionary values, including the lists inside. A `True` result
checks this one conversion against our hand-built example; it does not validate every label in the file:


In [13]:
# record_from_dev(): the hand-built record above, as a function of one development question
def record_from_dev(question):
    return {"question_id": question["question_id"], "split": "development", "question": question["question"],
            "type": "answerable", "known_support": question["known_support"],
            "references": sorted({a["text"] for a in question["answers"]}), "refuse": False}

record_from_dev(tesla) == record

True

Choose seven development questions that differ from one another:

- the Tesla year, and a second question answered by a single year (when Tesla emigrated);
- the Super Bowl MVP, whose known paragraph ranked second in the saved search;
- one with two accepted answers, *8* and *eight*;
- one with a long reference, the Vistula's animals;
- one with two different references, the meaning of *Norman*;
- the complexity question, to test whether the answer gives the requested measure rather than just naming a field.

This is a deliberately varied teaching sample, not a random estimate of performance. Each is found by the start of
its text. The loop appends one dictionary to `records` at a time, preserving the order in `chosen`; we will rely on
that order when we pair questions with results:


In [14]:
# chosen: the start of each question's text; records: the seven golden records, in that order
chosen = ["In what year was Nikola Tesla born", "In what year did Nikola Tesla emigrate", "Who was the Super Bowl 50 MVP",
          "How many appearances have the Denver Broncos", "What animals does the Vistula",
          "What is the original meaning of the word Norman", "What measure of a computational problem"]
records = []
for start in chosen:
    question = next(q for q in dev if q["question"].startswith(start))
    records.append(record_from_dev(question))
for r in records:
    print(r["known_support"], r["references"], "|", r["question"])

['S03-P00'] ['1856'] | In what year was Nikola Tesla born? 
['S03-P01'] ['1884'] | In what year did Nikola Tesla emigrate to the United States?
['S00-P02'] ['Miller', 'Von Miller'] | Who was the Super Bowl 50 MVP?
['S00-P01'] ['8', 'eight'] | How many appearances have the Denver Broncos made in the Super Bowl?
['S01-P02'] ['otter, beaver and hundreds of bird species', 'otter, beaver and hundreds of bird species.'] | What animals does the Vistula river's ecosystem include?
['S02-P02'] ['Norseman, Viking', 'Viking'] | What is the original meaning of the word Norman?
['S04-P01'] ['if its solution requires significant resources', 'its solution requires significant resources'] | What measure of a computational problem broadly defines the inherent difficulty of the solution?


Seven records, each with its paragraph and its accepted answers. The Vistula and complexity references come in two
wordings that differ by a full stop or a first word; the Norman question has two different answers.

The development questions are all answerable: each has a paragraph that holds its answer. A RAG system also meets
questions it must refuse and questions whose answer depends on a condition, and an evaluation set that leaves them out
cannot tell you how the system handles them. The next three records are about the five Atlas and Boreal policies in
`data/support-case.json`:

In [15]:
# policies: the five Atlas and Boreal policies, two sentences each
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
for policy in policies:
    print(policy["source_id"], "|", policy["text"])

H01 | Atlas passwords can be reset by email. Temporary accounts require a support agent.
H02 | Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
H03 | Atlas exports reports as CSV files. Only workspace owners can export all reports.
H04 | Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
H05 | Atlas invoices are sent monthly. Workspace owners can download invoices from Billing.


The policies say something about a product under specific conditions. H01's second sentence changes the normal
email route for temporary accounts; H04 belongs to Boreal and cannot supply an Atlas rule. Read the full policy before
choosing a reference.

Write three records by hand, each testing behaviour the seven development questions leave untested:

- G02, a **paraphrase**: the reset-link question in words that share few of H02's words; the answer is *30 minutes*.
- G05, an **exception**: the usual route is email, but H01's second sentence says temporary accounts need a support
  agent.
- G04, **out of scope**: a question no policy answers, so the right reply is *Insufficient evidence.*


In [16]:
# records: the seven development records plus three policy records written by hand
records += [
    {"question_id": "G02", "split": "development", "question": "When does the password recovery email stop working for Atlas?",
     "type": "paraphrase", "known_support": ["H02"], "references": ["30 minutes"], "refuse": False},
    {"question_id": "G05", "split": "development", "question": "Can temporary Atlas accounts use the usual email route?",
     "type": "exception", "known_support": ["H01"], "references": ["support agent"], "refuse": False},
    {"question_id": "G04", "split": "development", "question": "What is tomorrow's weather in Amman?",
     "type": "out_of_scope", "known_support": [], "references": [], "refuse": True},
]
print(len(records), "records |", sum(r["refuse"] for r in records), "to refuse")

10 records | 1 to refuse


The exception's reference is *support agent*: this record asks the reply to give the customer a usable route, beyond
answering yes or no. That is a decision about completeness in the answer key. A correct *No* alone will fail this
reference check. Write such requirements before reading the model's answer, so the scoring rule does not move to fit it.

`records += [...]` extends the existing list with three dictionaries. Run the construction cells in order; re-running
only that append cell adds the same cases again. If the count is no longer ten, rebuild `records` from the seven
questions, then append the policies once. G02, G05 and G04 are the IDs these questions keep in the larger golden set the later labs
use.

A set you score against must not change unnoticed. **Versioning** the set does that: the records are saved under a
hash of their text, a short fingerprint that changes when any record changes, so every saved score names the exact set
it was scored on, and two scores are compared only when they come from the same set. The subsection after this
checkpoint does it (self-study); Part D does not need it.

**Lesson to Remember**
> A golden record says what a right reply must contain, or that the reply must be a refusal, before any answer exists.
> The kinds of question in the set decide which failures it can find.

### Checkpoint C

| You should see | Our run |
|---|---|
| The Tesla record | `references: ['1856']`, `known_support: ['S03-P00']`, `refuse: False`; `record_from_dev(tesla) == record` is `True` |
| Ten records | seven from the development questions, three policy records, 1 to refuse |


### Fingerprint the records, instructions and corpus (self-study)

Save the ten records with three hashes, the records', the answer rules' (`RULES`, Lab 02's system message) and the
data files', because all three decide the scores, and with the stamp from Config. A **SHA-256 hash** turns any text
into 64 hexadecimal characters: the same text always gives the same hash, and a one-character change gives a
different one in practice. The first 12 characters make a convenient label for these small lab sets. Truncation
permits collisions, so this is not a guarantee of uniqueness or a security check. Lab 04's *Active* and *Inactive*
strings showed the mechanism on a visible one-word change. Here we use Python's public `hashlib.sha256()` API on
three inputs:

| Hash | Text being fingerprinted | Change it helps us notice |
|---|---|---|
| `records` | The ten records serialized as JSON | A changed question, reference or refusal requirement |
| `rules` | The `RULES` system instructions | A changed answering instruction |
| `data` | The chunks and policies files joined as text | A changed searchable corpus |

The helper `sha()` only joins the encode, hash and shorten operations; it does not save anything. Inspect the input
expressions in `version` as well as the returned labels:


In [17]:
# sha(): the first 12 hex characters of a text's SHA-256 hash
# version: hashes of the records, the answer rules and the two data files, plus the models and endpoints
import hashlib
from shared.answer import RULES

def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

data_text = "".join((DATA / name).read_text(encoding="utf-8") for name in ("squad-chunks.json", "support-case.json"))
version = {"records": sha(json.dumps(records, sort_keys=True)), "rules": sha(RULES), "data": sha(data_text),
           **stamp(settings)}
{name: version[name] for name in ("records", "rules", "data")}

{'records': 'b5789bcc0fdd', 'rules': '4be8e10f41b6', 'data': '5d58158d0880'}

### Read what each fingerprint covers

1. `json.dumps(records, sort_keys=True)`: the records as one text, keys in a fixed order, so the same records always
   give the same text with these serialization settings. Key order is fixed, but list order still matters. Reordering
   records changes the hash even when their contents are unchanged.
2. `.encode("utf-8")` turns the text into bytes, `hashlib.sha256(...).hexdigest()` hashes them into 64 hex characters,
   and `[:12]` keeps the first 12.
3. `rules` hashes `RULES`; `data` hashes the chunks file and the policies file read as one text.
4. `**stamp(settings)` adds the models, endpoints and run date from Config to the same dictionary.

Save the records and their version, named by the records' hash:


In [18]:
# path: .cache/golden-<records hash>.json, outside the corpus the pipeline searches
path = ROOT / ".cache" / f"golden-{version['records']}.json"
path.parent.mkdir(exist_ok=True)
path.write_text(json.dumps({"version": version, "records": records}, indent=1, ensure_ascii=False), encoding="utf-8")
print(path.relative_to(ROOT))

.cache/golden-b5789bcc0fdd.json


The file goes to `.cache/`, outside the index: a golden record inside the searchable corpus would be retrieved and
quoted by the pipeline it is meant to test. Change one record and the file name changes with it.

| You should see | Our run |
|---|---|
| Hashes | records `b5789bcc0fdd`, rules `4be8e10f41b6`, data `5d58158d0880` |
| Saved | `.cache/golden-b5789bcc0fdd.json` |

The fingerprints establish which serialized inputs we used. They do not check whether the labels are right, and
they do not identify every implementation choice: for example, the metric function's source is not hashed. Keep the
scoring code and settings comparable as well as checking the set version.


---

## Part D: Trace one evaluated answer, then score all ten questions

Part D produces one row per record: where the support was retrieved, what the model answered, and each measure. It
uses `records`, the ten records of Part C (the seven development questions, then G02, G05 and G04), and the two
functions of Part B.


We need a baseline whose retrieval and answer steps stay fixed while we learn to measure them. Lab 05 used
`OpenAIEmbeddings` directly on one question and two policy texts, then constructed an `InMemoryVectorStore`, added
documents and called `similarity_search_with_score()`. That demonstration comes before the configuration helpers we
reuse here. If those public methods are unfamiliar, work through Lab 05 Parts B–D first; a source listing alone is
not a demonstration of their inputs and outputs.

This lab measures the pipeline Lab 05 rebuilt with those components. It lives in `shared/lc_pipeline.py`, and the four
functions below are the whole pipeline, from the corpus to a cited answer (`load_corpus()` and `to_documents()` only
read the 277 records and wrap each one as a LangChain `Document`):

- `build_store()` takes the documents and embeds them into an `InMemoryVectorStore`, a fresh one each run; nothing is
  saved to disk.
- `retrieve()` takes the store and a question and returns the k closest documents with their cosine scores, best
  first; k is 3 by default.
- `evidence()` turns those hits into `[ID] text` lines for the model, every hit, with no token budget (Lab 02's
  `pack()` stopped at 400 tokens).
- `answer()` sends the question and the lines in one chat call and returns a `CitedAnswer`: Lab 02's answer shape
  written as a Python class, with `answer` (the reply text) and `claims` (each with `claim`, `source_id` and `quote`).
  `with_structured_output()` sends that shape as the reply format (a strict schema or plain JSON, as `JSON_MODE` says)
  and builds the object from the reply, so `reply.answer` is the answer text. One call, no retry. A valid object has
  the required fields and types; that validation does not establish that a claim is supported or a quote is correct.

Read the source once and match it to the public operations above: `build_store()` constructs the store and calls
`add_documents()`; `retrieve()` delegates to `similarity_search_with_score(..., k=3)`. The helpers keep these choices
consistent across evaluations. They do not introduce different retrieval or embedding algorithms:


In [19]:
# The four functions the rest of this lab calls, printed from shared/lc_pipeline.py so you see what each one does before you run it.
import inspect
from shared import lc_pipeline

for function in (lc_pipeline.build_store, lc_pipeline.retrieve, lc_pipeline.evidence, lc_pipeline.answer):
    print(inspect.getsource(function))


def build_store(documents: list[Document], settings: dict) -> InMemoryVectorStore:
    """An InMemoryVectorStore holding the documents, embedded with EMBED_MODEL through lc_embeddings()."""
    store = InMemoryVectorStore(lc_embeddings(settings))
    store.add_documents(documents)
    return store

def retrieve(store: InMemoryVectorStore, question: str, k: int = 3, filter=None) -> list:
    """The k closest documents, best first, as (Document, cosine score) pairs; filter is a function of a Document."""
    return store.similarity_search_with_score(question, k=k, filter=filter)

def evidence(hits: list) -> list[str]:
    """The hits as "[ID] text" lines, the format shared.answer.pack() writes and quote_found() reads."""
    return [f"[{document.id}] {document.page_content}" for document, _ in hits]

def answer(question: str, lines: list[str], settings: dict, rules: str = RULES) -> CitedAnswer:
    """One chat call through lc_chat_model() that returns a CitedAnswer (JSON_MODE picks a str

Follow the store cell from the inside out: `load_corpus()` returns a list of 277 dictionaries;
`to_documents()` returns a list of 277 LangChain `Document` objects in the same order; `build_store()` embeds their
text and returns one `InMemoryVectorStore` object. Each document keeps its chunk ID in `id`, text in `page_content`
and the source ID and other fields in `metadata`.

The index contains source text, not our evaluation records. Building it does not send the reference answers to the
model. Build this store once and reuse it for every record:


In [20]:
# store: the 272 chunks and the five policies as LangChain Documents, embedded into a fresh InMemoryVectorStore
store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
print(len(store.store), "documents in the store")

277 documents in the store


The 277 entries account for 272 chunks and five policies. `store.store` is the store's dictionary, keyed by document
ID. The count confirms how many entries were indexed; it does not check that the correct text belongs to each ID.

Score one record by hand, the paraphrase G02. It is the eighth record, `records[7]`, since positions count from 0:

> "When does the password recovery email stop working for Atlas?"

Its answer key is `known_support: ['H02']`, `references: ['30 minutes']`, `refuse: False`. Retrieval reads only
`record['question']`; we consult those labels after retrieving and answering.

*Predict first:* H02 answers it. At which rank will H02 come back, and what F1 will a right answer written as a
sentence get against *30 minutes*?


In [21]:
# record: G02; hits: the three closest documents and their cosine scores, best first
record = records[7]
hits = lc_pipeline.retrieve(store, record["question"])
for document, score in hits:
    print(document.id, "| source", document.metadata["source_id"], "| score", round(score, 4))

H01 | source H01 | score 0.5888
H04 | source H04 | score 0.4686
H02 | source H02 | score 0.4313


Each hit is a pair: a LangChain `Document` and its score. `document.id` is the chunk ID, and
`document.metadata["source_id"]` is the source it was cut from; for a policy the two are the same. The record's
support is written in source IDs, so look the rank up on the source IDs. The list comprehension keeps the retrieved
order; `_` receives the cosine score we do not need for this calculation. Matching a paragraph ID is a label match,
not a check that the particular retrieved chunk contains the answer:


In [22]:
# sources: the hits' source IDs in ranked order; then the rank of the first one in G02's known support
sources = [document.metadata["source_id"] for document, _ in hits]
print("retrieved:", sources, "| first support at rank", first_rank(sources, record["known_support"]))

retrieved: ['H01', 'H04', 'H02'] | first support at rank 3


G02's H02 support is present at rank 3. Now inspect the actual text the answering step receives. `evidence()` returns
a list of three strings, one per hit, preserving rank order. `"\n".join(lines)` puts them on separate display lines;
the same lines are included in the model's user message with the question.

`answer()` returns a `CitedAnswer` object, rather than a JSON string or dictionary. Its `.answer` field is a string;
`.claims` is a list of claim objects, each with `.claim`, `.source_id` and `.quote`. This cell scores only `.answer`.
Neither EM nor F1 reads the claims, verifies their quotations or checks factual support. Read the printed evidence
before comparing the generated sentence with `['30 minutes']`:


In [23]:
# lines: the evidence the model reads; reply: a CitedAnswer, whose .answer is the answer text
lines = lc_pipeline.evidence(hits)
print("\n".join(lines))
reply = lc_pipeline.answer(record["question"], lines, settings)
print("\nanswer:", reply.answer)
print("EM, F1:", em_f1(reply.answer, record["references"]))

[H01] Atlas passwords can be reset by email. Temporary accounts require a support agent.
[H04] Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.



answer: The password recovery email for Atlas stops working after 30 minutes.
EM, F1: (0, 0.33333333333333337)


H02, the policy that answers, comes third, behind H01 and the Boreal policy H04: a hit at 3, with a reciprocal rank
of 1/3. The answer is right, *The password recovery email for Atlas stops working after 30 minutes.*, and scores F1
about 0.33: its two reference words are two of its ten, so precision is 0.2 and recall 1. (`0.33333333333333337` is
the same number with every digit a Python float keeps.)

Now the same steps for every record, as one function that returns a row. A refusal record gets no EM or F1, because
there is no reference to compare with; its row records whether the model said *Insufficient evidence.* Keep these
two fields separate: `refuse` is the expected behaviour from the record, while `refused` is the observed reply.
We compare them when reporting refusals.


In [24]:
# score_row(): retrieve, answer and score one record; returns its row as a dictionary
INSUFFICIENT = "Insufficient evidence."

def score_row(record):
    hits = lc_pipeline.retrieve(store, record["question"])
    reply = lc_pipeline.answer(record["question"], lc_pipeline.evidence(hits), settings)
    sources = [document.metadata["source_id"] for document, _ in hits]
    row = {"id": record["question_id"], "type": record["type"], "refuse": record["refuse"],
           "rank": first_rank(sources, record["known_support"]),
           "answer": reply.answer, "refused": reply.answer.strip() == INSUFFICIENT}
    if not record["refuse"]:
        row["em"], row["f1"] = em_f1(reply.answer, record["references"])
    return row

### Keep expected behaviour and observed behaviour in one result row

1. `retrieve()` and `answer()`: the three hits and the model's reply, as for G02.
2. `sources` and `first_rank()`: the rank of the first known source, or `None`.
3. `row`: the record's ID, its type and whether it should be refused; the rank; the answer; and `refused`, whether the
   answer is exactly *Insufficient evidence.* (`strip()` drops spaces at the ends).
4. `em` and `f1`, only for a record that expects an answer.

`score_row()` returns a dictionary. It reads the shared `store` and `settings` already built in this notebook.
Its compact row is enough for these measures; Part F's reusable scorer also keeps the evidence and claims.

Run it on the ten records, one chat call each. Each line shows the start of the question, the rank, the scores and the
answer. `rows[i]` belongs to `records[i]`: the comprehension preserves input order, and `zip(records, rows)` pairs
those positions. If you sort just one list, the pairs will be wrong. The question is shortened to 36 characters only
for printing; the full text was used for the request:


In [25]:
# rows: one row per record, in the order of records
rows = [score_row(record) for record in records]
for record, row in zip(records, rows):
    scores = f"EM {row['em']}  F1 {row['f1']:.3f}" if not row["refuse"] else "refusal record"
    print(f"{record['question'][:36]:38}rank {row['rank']!s:5}{scores:17}| {row['answer']}")

In what year was Nikola Tesla born?   rank 1    EM 0  F1 0.250   | Nikola Tesla was born in the year 1856.
In what year did Nikola Tesla emigra  rank 1    EM 1  F1 1.000   | 1884
Who was the Super Bowl 50 MVP?        rank 2    EM 0  F1 0.444   | Von Miller was the Super Bowl 50 MVP.
How many appearances have the Denver  rank 1    EM 0  F1 0.200   | The Denver Broncos have made eight appearances in the Super Bowl.
What animals does the Vistula river'  rank 1    EM 0  F1 0.778   | The Vistula river's ecosystem includes the otter, beaver, and hundreds of bird species.
What is the original meaning of the   rank 1    EM 0  F1 0.333   | The original meaning of the word Norman is "Northman" or "Norseman, Viking."
What measure of a computational prob  rank 1    EM 0  F1 0.000   | Computational complexity theory broadly defines the inherent difficulty of a computational problem.
When does the password recovery emai  rank 3    EM 0  F1 0.333   | The password recovery email for Atlas stops workin

Read the ten rows column by column:

- **rank**: every record with support found it in the top three. The MVP's paragraph came second and G02's policy
  third; G04 shows `None` because it has no support to find.
- **EM**: 1 only for *1884*, the one answer given as a bare year. Every other answer is a sentence, and a sentence is
  different from its short reference in these saved rows.
- **F1**: seven answers meet the requested content on reading the questions and evidence. They score from 0.2
  (the Broncos' *eight appearances*) to 1 (the bare year), according to word overlap. That judgement comes from
  reading the replies, not from an F1 threshold.
- Two answers score 0.000, and both miss what their record asks for. The complexity answer names the field
  (*computational complexity theory*) instead of the measure the question asks about. G05's *No* is true, but it
  leaves out the support agent a temporary-account customer needs. Both have their support at rank 1.
- G04, the weather question, was refused with *Insufficient evidence.*, as its record asks.

The retrieved evidence contains the missing information for both failures at rank 1. Changing retrieval alone is
therefore not an explanation for these two replies. We would next inspect the answering instructions and how the
model used the evidence. Keep that diagnosis beside the score before reducing the rows to an average.

**Lesson to Remember**
> One row per question keeps the retrieval, the answer and the scores side by side, so every number can be checked
> against the answer it came from. Read the rows before you average them.

### Checkpoint D

| You should see | Our run |
|---|---|
| The store | 277 documents |
| G02 by hand | H01, H04, H02 (scores about 0.59, 0.47, 0.43); first support at rank 3; *... stops working after 30 minutes.*; EM 0, F1 0.333 |
| Ten rows | rank 1 except the MVP (2), G02 (3) and G04 (`None`); EM 1 only for *1884*; F1 0.000 for the complexity answer and G05; G04 *Insufficient evidence.* |


---

## Part E: Choose checks and denominators that match the expected behaviour (self-study)

Part E turns the rows into the few numbers you would report, and checks each number against the rows. Its last
cell prints the records' hash from the versioning subsection of Part C: if you skipped that subsection, run its
two cells first.

Retrieval measures need known support. G04 has none, since no source answers it, so it stays out of hit@3 and MRR, and
the line says so. An unanswerable case with no support is not a retrieval miss against a known label. In contrast,
an answerable record whose labelled source is absent remains in this denominator with zero credit. The formulas are
Part A's, with $Q$ the records that have known support. `zip(rows, records)` depends on their unchanged positional
alignment from Part D:


In [26]:
# with_support: the rows of records that name a known source; hit@3 and MRR@3 over those rows only
with_support = [row for row, record in zip(rows, records) if record["known_support"]]
hits3 = sum(row["rank"] is not None for row in with_support)
mrr = sum(1 / row["rank"] if row["rank"] else 0 for row in with_support) / len(with_support)
print(f"hit@3: {hits3} of {len(with_support)} records with known support ({len(rows) - len(with_support)} without, not counted)")
print(f"MRR@3: {mrr:.3f} over the same {len(with_support)}")

hit@3: 9 of 9 records with known support (1 without, not counted)
MRR@3: 0.870 over the same 9


Every record with support found it in the top three: hit@3 9 of 9 in our run. MRR@3 is about 0.87, lower, because
two records found their support later, the MVP at rank 2 and G02 at rank 3: (7 × 1 + 1/2 + 1/3) / 9. hit@3 reads as
perfect recovery of the labelled sources at depth 3; MRR shows that two were ranked later. Neither score tests how
the model used them, and neither checks whether another unlabelled passage could answer.

The answer measures cover the records that expect an answer, $A$, the refusal record left out because it has no
reference:

$$
\text{EM count} = \sum_{a \in A} \text{EM}_a \qquad \overline{F_1} = \frac{1}{|A|} \sum_{a \in A} F_{1,a}
$$

- $\text{EM}_a$, $F_{1,a}$: the scores in answer record $a$'s row.
- $\overline{F_1}$: the arithmetic mean of the per-record F1 values, giving each answer record equal weight.


In [27]:
# answer_rows: the rows that expect an answer; the EM count and the mean F1 over them
answer_rows = [row for row in rows if not row["refuse"]]
print(f"EM: {sum(row['em'] for row in answer_rows)} of {len(answer_rows)} answer records")
print(f"F1: {sum(row['f1'] for row in answer_rows) / len(answer_rows):.3f}, the mean over the same {len(answer_rows)}")

EM: 1 of 9 answer records
F1: 0.371, the mean over the same 9


An average hides which answer produced which score. `sorted(answer_rows, key=lambda row: row['f1'])` makes a new
list ordered by each dictionary's F1 value; `lambda` supplies that one-field sorting rule. It leaves `rows` and
`records` in their original aligned order. Read the lowest F1 rows first:


In [28]:
# the answer rows sorted by F1, lowest first, with the rank of their support
for row in sorted(answer_rows, key=lambda row: row["f1"]):
    print(f"F1 {row['f1']:.3f}  rank {row['rank']}  | {row['answer']}")

F1 0.000  rank 1  | Computational complexity theory broadly defines the inherent difficulty of a computational problem.
F1 0.000  rank 1  | No, temporary Atlas accounts cannot use the usual email route.
F1 0.200  rank 1  | The Denver Broncos have made eight appearances in the Super Bowl.
F1 0.250  rank 1  | Nikola Tesla was born in the year 1856.
F1 0.333  rank 1  | The original meaning of the word Norman is "Northman" or "Norseman, Viking."
F1 0.333  rank 3  | The password recovery email for Atlas stops working after 30 minutes.
F1 0.444  rank 2  | Von Miller was the Super Bowl 50 MVP.
F1 0.778  rank 1  | The Vistula river's ecosystem includes the otter, beaver, and hundreds of bird species.
F1 1.000  rank 1  | 1884


A mean F1 of about 0.37 is a mean word-overlap score. EM's 1 of 9 counts identical normalised texts. Reading either
as the fraction of factually correct answers would be a mistake. The sorted rows let us inspect what those averages hide: seven answers meet the requested content but spread from
0.2 to 1 in word overlap. The two that omit the required content, the complexity answer and G05, have support at
rank 1. A low F1 alone cannot distinguish a wrong fact from a correct answer with different or extra wording.

For a quick screening check on generated sentences, ask whether a whole accepted answer appears in the reply. Benchmarks of long
generated answers do this: ALCE, for example, checks whether each short reference answer is a substring of the
generated text. Here:

$$
\text{found}(a, R) = 1 \text{ when some } r \in R \text{ appears in } a \text{ as whole words, both normalised; else } 0
$$

- $R$: the accepted answers, as in Part B.
- whole words: a space goes at each end of both normalised texts, so *no* is not found inside *know*;
- `any(...)`: return `True` as soon as one reference matches, or `False` when none matches.

This is phrase presence. It has no understanding of negation, product names or paraphrases. A reply saying *Tesla was
not born in 1856* contains the reference *1856* and would pass. A correct synonym absent from the reference list can
fail. Keep the function small so its limits remain visible:


In [29]:
# found(): True when a whole reference, normalised, appears in the normalised answer; tried on two pairs
def found(prediction, references):
    answer_text = f" {normalize(prediction)} "
    return any(f" {normalize(reference)} " in answer_text for reference in references)

print(found("Nikola Tesla was born on July 10, 1856.", ["1856"]), found("I know the answer.", ["no"]))

True False


The results are booleans: the Tesla reference is present, and *no* is not a whole word in *know*. Now apply that same
check to the real answer rows. `answer_rows` contains the same dictionary objects as `rows`, so adding `row['found']`
through `rows` also makes the field available through `answer_rows`. We are updating those dictionaries, not copies:


In [30]:
# found on every answer record, stored in its row; then the answers it did not find
for row, record in zip(rows, records):
    if not record["refuse"]:
        row["found"] = found(row["answer"], record["references"])
print(f"found: {sum(row['found'] for row in answer_rows)} of {len(answer_rows)} answer records")
for row in answer_rows:
    if not row["found"]:
        print("  not found:", row["id"], "|", row["answer"])

found: 7 of 9 answer records
  not found: 56e16839cd28a01900c67887 | Computational complexity theory broadly defines the inherent difficulty of a computational problem.
  not found: G05 | No, temporary Atlas accounts cannot use the usual email route.


7 of 9 in our run, and the two not found are the two that miss what their record asks for: the complexity question
(shown by its SQuAD ID) and G05. On these rows the reference check agrees with a reader, where F1 could only rank.

Phrase presence does not tell us which product the time belongs to. Asked about Boreal (reference *10 minutes*),
this constructed answer includes both products' correct times:


In [31]:
# an answer that holds the reference and the other product's time beside it
found("Atlas: 30 minutes. Boreal: 10 minutes.", ["10 minutes"])

True

It passes because *10 minutes* is present. This particular answer gives both times correctly; the pass is not a
false positive. But reversing the product/time assignments to *Atlas: 10 minutes. Boreal: 30 minutes.* would still
contain the same accepted phrase. `found()` cannot distinguish those assignments. A product-aware check or a reader
must inspect the relationship; Lab 07 tests a judge against human labels for that reason.

The normalisation also matters: ASCII punctuation is removed, curly quotation marks remain, and plurals are not
reduced to singular forms. A failed phrase check is a row to inspect, rather than proof that the fact is wrong.

Last, refusals. They get their own lines, with their own denominators, both ways round:

- refused when it should: the refusal records refused, out of all refusal records;
- refused when it should answer: the answer records refused, out of all answer records.


In [32]:
# the two refusal counts, then the stamp line: model, endpoint and the golden records' hash
from shared.settings import chat_url

refusal_rows = [row for row in rows if row["refuse"]]
print(f"refused when it should:        {sum(row['refused'] for row in refusal_rows)} of {len(refusal_rows)} refusal records")
print(f"refused when it should answer: {sum(row['refused'] for row in answer_rows)} of {len(answer_rows)} answer records")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)} | golden records {version['records']}")

refused when it should:        1 of 1 refusal records
refused when it should answer: 0 of 9 answer records

gpt-4o-mini at https://api.openai.com/v1 | golden records b5789bcc0fdd


The weather question was refused (1 of 1), and no answer record was refused (0 of 9). One refusal record says little
about refusals; Part F scores a set with more.

**Lesson to Remember**
> State which records a metric can score, keep misses in that population, and print the denominator. Reference
> presence screens answer wording; refusal counts test the expected reply. Neither replaces reading factual support.

### Checkpoint E

| You should see | Our run |
|---|---|
| Retrieval | hit@3 9 of 9 (1 without support, not counted); MRR@3 0.870 |
| Answers | EM 1 of 9; F1 0.371; at 0.000 the complexity answer and G05's *No, temporary Atlas accounts ...* |
| `found()` | `True False` on the two test pairs; 7 of 9 rows; not found: the complexity question and G05; both products' times `True` |
| Refusals | refused 1 of 1 refusal records, 0 of 9 answer records |


---

## Part F: Find policy failures hidden by the whole-set score (self-study)

A **slice** is a subset chosen by a property such as question type or expected refusal. We need slices because the
whole-set count gives common cases more influence than rare ones. A pipeline can handle many fact questions and
still apply another product's policy.

The shared set contains all 30 development questions followed by eight constructed policy cases from
`06_Evaluation_Set/resources/golden-policy-cases.json`. G02, G05 and G04 are the cases we already wrote. The extra
cases test specific boundaries:

| Cases | What they ask us to check |
|---|---|
| G01 and G02 | The direct Atlas duration question and its paraphrase both need *30 minutes* |
| G03 | The answer step receives only H02, which gives a duration but cannot answer the temporary-account question |
| G04 and G07 | Refuse weather and an unsupported one-hour extension request |
| G05 | Keep H01's temporary-account exception and give the support-agent route |
| G08 | Refuse the Boreal export question because H03 describes Atlas, not Boreal |
| G09 | Answer the Atlas reset-method question from H01 even if Boreal's mobile-app policy ranks higher |

G03 has an extra field, `pack: ['H02']`. Its `known_support: ['H01']` says where the answer exists in the corpus;
`pack` says which document the test deliberately gives the answering step. It bypasses search to isolate refusal
when evidence is missing. A `None` rank on this case describes the fixed evidence, not a retrieval failure. Follow-up
questions are left to Lab 09, where the conversation is part of the input.

We have now written the metrics, converted records and scored a pipeline by hand. `shared/evals.py` packages those
steps so later labs use the same checks. Read its source and compare the function names with the cells you ran:


In [33]:
# the whole of shared/evals.py, the module Labs 07 to 11 use
import inspect
from shared import evals

print(inspect.getsource(evals))

"""Golden records and the measures that score a RAG pipeline on them: the steps Lab 06 builds by hand.

em_f1() and found() compare an answer with its reference answers; first_rank() finds the first retrieved source a
record names as support. record_from_dev() turns a development question into a golden record, and golden_set() reads
the whole development golden set: the 30 development questions and the reviewed policy cases in Lab 06's resources/.
score() runs one record through shared.lc_pipeline and returns its row; version() names the set, the rules and the
data a run used.
"""
from __future__ import annotations

import hashlib
import json
import re
import string
from collections import Counter

from shared.answer import RULES
from shared.lc_pipeline import answer, evidence, retrieve
from shared.settings import DATA, ROOT, stamp

REFUSAL = "Insufficient evidence."
POLICY_CASES = ROOT / "06_Evaluation_Set" / "resources" / "golden-policy-cases.json"


def normalize(text: str) -> str:


The module keeps the calculations we just built and joins them into a reusable scorer:

- `normalize()`, `em_f1()`, `found()`, `first_rank()` and `record_from_dev()` perform the operations from Parts A–E.
- `golden_set()` returns a list of 38 dictionaries: the 30 development records in file order, then the eight policy
  cases. The IDs skip G06; a case ID is a stable label, not the list position or a count.
- `score(record, store, settings)` returns one result dictionary. It retrieves three hits, or loads the document IDs
  named by `pack`, formats evidence and makes one answering call. It also retains `retrieved` chunk IDs, `evidence`
  strings and `claims` dictionaries. `claim.model_dump()` converts each Pydantic claim object into a plain dictionary.
- `version()` returns the record/rules/data hashes and model/endpoint stamp from Part C's versioning subsection.

The result's `question_id` corresponds to `id` in our smaller `score_row()`. `refuse` still holds the expected
behaviour; `refused` records the observed exact refusal. The new `passed` field chooses one check: for a refusal
record it equals `refused`; for an answer record it equals `found(answer, references)`. A pass does not verify the
claims or their quotes, and the two branches test different requirements.

The following comprehension returns `golden_rows[i]` for `golden[i]`. It scores all 38 records, one chat call each,
then selects the eight policy rows by IDs starting with G. `golden_rows[:30]` is valid for the development summary
because `golden_set()` put those records first:


In [34]:
# golden: the 38 records; golden_rows: one row each; policy_rows: the eight policy cases (IDs starting with G)
golden = evals.golden_set()
golden_rows = [evals.score(record, store, settings) for record in golden]
policy_rows = [row for row in golden_rows if row["question_id"].startswith("G")]
print(f"passed {sum(row['passed'] for row in golden_rows)} of {len(golden_rows)} records")
print(f"  the 30 development questions: {sum(row['passed'] for row in golden_rows[:30])} of 30")
print(f"  the {len(policy_rows)} policy cases:          {sum(row['passed'] for row in policy_rows)} of {len(policy_rows)}")

passed 34 of 38 records
  the 30 development questions: 29 of 30
  the 8 policy cases:          5 of 8


The screen reports 34 of 38 passing the selected checks: 29 of 30 development records and 5 of 8 policy cases.
Development records supply 30 of the 38 votes, so their phrase matches dominate the total. The smaller policy group
needs its own breakdown before we decide what to change.

Split the policy rows by `type`, then make a refusal slice using `refuse`. These are overlapping views: G08 belongs
to `wrong_product` and also to the refusal slice, so do not add all their denominators together. `dict.fromkeys(...)`
keeps each type once in first-seen order; the inner list comprehension selects the rows of that type:


In [35]:
# each policy type with its pass count; then the refusal slice: every record that should be refused
for kind in dict.fromkeys(row["type"] for row in policy_rows):
    kind_rows = [row for row in policy_rows if row["type"] == kind]
    print(f"{kind:22}{sum(row['passed'] for row in kind_rows)} of {len(kind_rows)}")
refusal_slice = [row for row in golden_rows if row["refuse"]]
print(f"\nrefusal slice: refused {sum(row['refused'] for row in refusal_slice)} of {len(refusal_slice)}")

answerable            1 of 1
paraphrase            1 of 1
missing_pack_support  1 of 1
out_of_scope          1 of 1
exception             0 of 1
missing_from_corpus   1 of 1
wrong_product         0 of 2

refusal slice: refused 3 of 4


The type counts show where to look, but they do not explain a failure. Print each row whose `passed` is false and
read its answer against its question, reference and evidence. `retrieved` contains chunk IDs; `rank` was computed
from their source IDs. For a fixed-pack case such as G03, those IDs name the supplied pack instead of search results:


In [36]:
# every failing row: its ID and type, the retrieved chunk IDs, the rank of its support, and the answer
for row in golden_rows:
    if not row["passed"]:
        print(f"failed {row['question_id']} ({row['type']}) | retrieved {row['retrieved']} | rank {row['rank']}")
        print("  ", row["answer"])

failed 56e16839cd28a01900c67887 (answerable) | retrieved ['S04-P01-C00', 'S04-P00-C00', 'S04-P02-C01'] | rank 1
   Computational complexity theory broadly defines the inherent difficulty of a computational problem.
failed G05 (exception) | retrieved ['H01', 'H02', 'H04'] | rank 1
   No, temporary Atlas accounts cannot use the usual email route.
failed G08 (wrong_product) | retrieved ['H03', 'H05', 'S38-P00-C01'] | rank None
   Only workspace owners can export all reports in Boreal.
failed G09 (wrong_product) | retrieved ['H04', 'H01', 'H02'] | rank 2
   Yes, password reset uses a mobile app.


The failed types are the exception (0 of 1) and wrong product (0 of 2); the overlapping refusal slice passes 3 of 4.
G08 asks about Boreal, but the only export policy is Atlas's H03, retrieved first. The saved reply is
*Only workspace owners can export all reports in Boreal.* It assigns Atlas's rule to Boreal. G08 has no reference or
known support, so answer F1 and known-source retrieval metrics cannot test it. Its required refusal makes the failure
visible.

G09 retrieves the correct Atlas source H01 second, behind Boreal's H04, yet answers with the mobile-app rule. It is
a known-source hit at 3 and a failed answer phrase check. G05's reply omits the required support-agent route. The
complexity question also fails the phrase check; read it beside its reference before deciding what repair to test.

Our first ten records contained only one required refusal, and it passed. The larger set exposes another kind of
required refusal that fails. Adding question types gave us information that simply repeating the weather question
would not provide.

One or two records in a slice are enough to show a concrete failure, but not to estimate how often it happens.
Repeat the eight policy cases twice more. `passes` starts with each first-run boolean converted to 0 or 1;
`+=` adds each later pass. The final dictionary counts three observations per case, not three different questions:


In [37]:
# passes: for each policy case, the runs it passed, starting from the run above; two more runs follow
policy_cases = [record for record in golden if record["question_id"].startswith("G")]
passes = {row["question_id"]: int(row["passed"]) for row in policy_rows}
for _ in range(2):
    for record in policy_cases:
        passes[record["question_id"]] += evals.score(record, store, settings)["passed"]
print("passed, of 3 runs:", passes)

passed, of 3 runs: {'G01': 3, 'G02': 3, 'G03': 3, 'G04': 3, 'G05': 0, 'G07': 3, 'G08': 0, 'G09': 0}


The same three cases failed in all three runs, G05, G08 and G09 (0 of 3 each), and the other five passed 3 of 3.
These observed failures give us specific cases to test a change against. Three passes or failures do not establish
a probability for future requests, and the repeated calls may produce different replies on your run.

Read these counts for what they are. The 38 records are 30 questions from five Wikipedia articles and eight cases
about five two-sentence policies, and `known_support` names the paragraph each question was written from, not every
passage that could answer it. The counts compare versions of this pipeline on the same records; they do not estimate
how often it will be right on the questions people actually ask.

Save this first 38-row evaluation with its version. The JSON has `version` and `rows`; `passes`, the three-run
summary, is printed above but is not included in that file. A later comparison should check the records, rules, data,
model and endpoints in the version metadata. A changed pipeline is what we intend to compare; a changed answer key
needs separate treatment.

The filename includes the model and records hash. It does not include all the version fields and is not a unique
run ID: running the same cell again can overwrite that path. Read the metadata inside the file before reusing it:


In [38]:
# path: .cache/golden-rows-<model>-<records hash>.json, the rows of this run for later labs
golden_version = evals.version(golden, settings)
model = settings["CHAT_MODEL"].replace("/", "_")
path = ROOT / ".cache" / f"golden-rows-{model}-{golden_version['records']}.json"
path.write_text(json.dumps({"version": golden_version, "rows": golden_rows}, indent=1, ensure_ascii=False),
                encoding="utf-8")
print(path.relative_to(ROOT), "|", len(golden_rows), "rows")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

.cache/golden-rows-gpt-4o-mini-42126604e98f.json | 38 rows

gpt-4o-mini at https://api.openai.com/v1


The printed relative path confirms the file was written with 38 rows. It does not certify the facts in those rows.
Labs 07 and 11 load the question set with `evals.golden_set()` and score a change with `evals.score()`; Lab 11 turns the
refusal slice into a separate release requirement, so a high overall average cannot compensate for a failed refusal.

**Lesson to Remember**
> A count over the whole set is dominated by the kind of question it holds most of. Report each slice with its own
> denominator, and run a small slice more than once before you trust it.

### Checkpoint F

| You should see | Our run |
|---|---|
| Whole set | passed 34 of 38; development questions 29 of 30; policy cases 5 of 8 |
| Slices | exception 0 of 1, wrong_product 0 of 2, the other types 1 of 1; refusal slice 3 of 4; failed: the complexity question, G05, G08, G09 |
| Three runs | G05, G08 and G09 0 of 3; the other five 3 of 3 |
| Saved | `.cache/golden-rows-gpt-4o-mini-42126604e98f.json`, 38 rows |


---

## What you can do now

- Compute hit@3 and MRR@3 from ordered source labels, including misses, and explain the limits of a source match.
- Compute EM and word-token F1 against multiple references and explain why neither is factual accuracy.
- Write evaluation records whose labels specify accepted content or required refusal, then fingerprint the inputs.
- Run the same pipeline on those records and keep every result connected to its question and answer key.
- Choose denominators for retrieval, answers and refusals, then inspect type slices and individual failures before
  accepting a whole-set score.

The next challenge adds a pair of cases that differ in whether the corpus actually supports an answer. Decide the
expected behaviour from the policies first; let that decision determine the score, rather than the reply's fluency.


## Challenge: Test Boreal billing and reset questions

Use the policies printed in Part C and the scorer from Part F. Task 1 is the challenge; Task 2 is optional. Save the
printed results and explain what the checks establish. The worked solution has example counts, but your run can differ.

1. **Write two records before scoring them.** Complete `billing` and `reset` under Task 1 for *Where do Boreal
   workspace owners download invoices?* and *How long is a Boreal reset link valid?* Review H05's product and H04's
   duration. For each record, choose a type, known support, references and whether refusal is required. Keep list
   fields as lists, including when empty. The supplied loop scores each record three times.

   **Check:** read the pass count and all three answers for each record. In the worked solution the billing case
   passes 0 of 3: each answer uses Atlas's Billing rule for a Boreal request. The reset case passes 3 of 3 and gives
   *10 minutes*, with support at rank 1. These are observations from the saved solution, not guaranteed outcomes.

   **Explain:** suppose you incorrectly labelled the billing case answerable, with H05 as support and *Billing* as
   the reference. Which retrieval and phrase checks could then reward the wrong-product reply? Would EM or F1
   establish that the rule belongs to Boreal? Which field selects the refusal branch of `evals.score()`, and why
   must the other labels agree with that choice?

2. **Optional: version an expanded reference list.** In a copy of the shared set, add *half an hour* to G01's
   references while retaining *30 minutes*.

   **Check:** print the old and new records hashes and rescore the same G01 answer text with each reference list.
   The worked solution changes `42126604e98f` to `597edd27f28d`. Its answer *Atlas reset links expire after 30
   minutes.* keeps F1 0.444 because *30 minutes* remains the best-matching reference.

   **Explain:** the same score on this one answer does not mean both sets apply the same scoring conditions.
   Why should a pipeline comparison rescore both versions on one agreed reference set? Which kinds of reply
   could benefit from the extra accepted wording?


**Task 1.** Complete the four label fields in each dictionary before running the cell. `None` marks an unfinished
field; an empty list is a deliberate label and is different from `None`. The guard only detects unfinished fields,
not incorrect labels. Once complete, the nested loop makes six chat calls.

For each case, `runs` is a list of three result dictionaries. Summing the boolean `passed` fields counts successes.
The printed rank is only `runs[0]['rank']`; read all three answers rather than assuming they are identical.


In [39]:
billing = {"question_id": "C01", "split": "development", "question": "Where do Boreal workspace owners download invoices?",
           "type": None, "known_support": None, "references": None, "refuse": None}   # TODO
reset = {"question_id": "C02", "split": "development", "question": "How long is a Boreal reset link valid?",
         "type": None, "known_support": None, "references": None, "refuse": None}       # TODO

if None in billing.values() or None in reset.values():
    print("⚠️ Complete both records, then run the cell again.")
else:
    for record in (billing, reset):
        runs = [evals.score(record, store, settings) for _ in range(3)]
        print(f"{record['question_id']} passed {sum(row['passed'] for row in runs)} of 3 | rank {runs[0]['rank']}")
        for row in runs:
            print("  ", row["answer"])

⚠️ Complete both records, then run the cell again.


**Task 2 (optional).** `copy.deepcopy(golden)` makes a separate copy of the list and its nested dictionaries and
reference lists. A shallow list copy would still share the records, so changing a reference could silently change
the original set too. `next(...)` finds G01 by its stable ID rather than assuming its list position.

Add the reference to `g01['references']`, keeping the existing reference. This task makes no model call: it hashes
the two sets and recalculates F1 for the G01 answer already held in `golden_rows`. The guard checks that the original
one-reference list changed; you must still inspect that the intended wording was added.


In [40]:
import copy

changed = copy.deepcopy(golden)
g01 = next(record for record in changed if record["question_id"] == "G01")
# TODO: add "half an hour" to g01["references"]

if g01["references"] == ["30 minutes"]:
    print("⚠️ Add the reference, then run the cell again.")
else:
    print("records hash before:", evals.version(golden, settings)["records"],
          "| after:", evals.version(changed, settings)["records"])
    answer_text = next(row["answer"] for row in golden_rows if row["question_id"] == "G01")
    print("G01 F1, old references:", round(evals.em_f1(answer_text, ["30 minutes"])[1], 3),
          "| new references:", round(evals.em_f1(answer_text, g01["references"])[1], 3), "|", answer_text)

⚠️ Add the reference, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ZeroDivisionError: division by zero` in an F1 function | precision divided by the answer's length before checking the overlap, on an empty answer | Test `if overlap` before dividing, as `em_f1()` does |
| hit@3 is 0 for every development record | the documents' IDs (chunk IDs, `S03-P00-C00`) compared with `known_support` (source IDs, `S03-P00`) | Compare `document.metadata["source_id"]` |
| `TypeError: unsupported operand type(s) for /: 'int' and 'NoneType'` | `1 / row["rank"]` on a record whose support was not retrieved | Write `1 / rank if rank else 0`, and keep the record in the denominator |
| `KeyError: 'f1'` | an average over every row, refusal records included | Average over the answer records only, and give refusals their own line |

## Resources

- [Rajpurkar et al., *SQuAD: 100,000+ Questions for Machine Comprehension of Text*, 2016](https://arxiv.org/abs/1606.05250):
  the dataset, its several answers per question, and the EM and F1 its scoring script computes
- [Voorhees, *The TREC-8 Question Answering Track Report*, 1999](https://trec.nist.gov/pubs/trec8/papers/qa_report.pdf):
  the question-answering evaluation that scored systems by mean reciprocal rank
- [Gao et al., *Enabling Large Language Models to Generate Text with Citations* (ALCE), 2023](https://arxiv.org/abs/2305.14627):
  checking short reference answers inside long generated answers, and citation quality
- [LangSmith: evaluation concepts](https://docs.langchain.com/langsmith/evaluation-concepts): datasets of examples,
  reference outputs, evaluators and experiments, the same ideas as a hosted service
- [Python `collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter) and
  [`hashlib`](https://docs.python.org/3/library/hashlib.html): the word counts behind token F1, and the hashes that
  version a set

Next: [Lab 07](../07_LLM_Judge_Calibration/README.md), *How to check an LLM judge against your own labels*. Back to [the lab list](../README.md).